---

## **DIPLOME UNIVERSITAIRE**

## **Sorbonne Data Analytics**

## **Projet Generative AI**

## **Système Agentique d'Évaluation et d'Anticipation des Risques Climatiques et Hydrologiques**

## **SAEARCH**




Promotion 007

Avril 2026




**Dataset** : EM-DAT — Decadal Average Annual Number of Deaths from Disasters (1900–2020)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible. Si aucune sortie naturelle, ajouter un `print()` de vérification.

---

# NOTEBOOK 10 — ML Prédictif : Mortalité liée aux Catastrophes Climatiques

**Auteur :** Xia Bizot

---

### Objectif

Entraîner un modèle de Machine Learning sur les données EM-DAT (1900–2020) pour prédire la impact humain des catastrophes climatiques par pays et par décennie. Pipeline complet : EDA, feature engineering, comparatif multi-modèles, tuning, MLflow tracking, prédictions 2030.

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, seed |
| 2. Chargement & EDA | Exploration, distributions, top pays, tendances |
| 3. Feature Engineering | Lag, trend, volatilité, continent, log-transform |
| 4. Préprocessing | Train/test split, scaling, encodage |
| 5. Modèles de Régression | Baseline → RF → GBM → XGBoost, comparatif |
| 6. Modèles de Classification | Niveaux de risque, comparatif |
| 7. Hyperparameter Tuning | GridSearchCV sur le meilleur modèle |
| 8. MLflow Tracking | Log params, metrics, artifacts |
| 9. Prédictions 2030 | Top pays à risque, cartographie |
| 10. Intégration Agent | Le modèle comme complément au RAG |
| 11. Conclusions | Quality gate, limites, axes d'amélioration |

---

### Hypothèse testable

> Un modèle de ML entraîné sur les données historiques EM-DAT (1900–2020) peut prédire le niveau de niveau d'exposition par catastrophe climatique d'un pays pour la décennie 2030, avec un R² > 0.5 en régression et un F1 > 0.6 en classification.

---

---

## 1. Configuration

In [ ]:
import warnings
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, f1_score, accuracy_score,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

SEED = 42
np.random.seed(SEED)
notebook_start_time = time.time()

BASE = Path('..')
DATA_PATH = BASE.parent / 'decadal-average-annual-number-of-deaths-from-disasters.csv'
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Palette
COLOR = 'steelblue'
COLORS = ['steelblue', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = (10, 5)

assert DATA_PATH.exists(), f'CSV introuvable : {DATA_PATH}'
print(f'  XGBoost : {"OK" if HAS_XGB else "non install\u00e9"}')
print(f'  MLflow  : {"OK" if HAS_MLFLOW else "non install\u00e9"}')
print('>> 1. Configuration : OK')

---

## 2. Chargement & EDA

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = ['country', 'year', 'impact']

print(f'Shape brute : {df_raw.shape}')
print(f'Pays/entit\u00e9s uniques : {df_raw["country"].nunique()}')
print(f'D\u00e9cennies : {sorted(df_raw["year"].unique())}')
print(f'Valeurs nulles : {df_raw.isnull().sum().sum()}')
print(f'\nStatistiques target :')
print(df_raw['impact'].describe())

# Filtrer les agr\u00e9gats (garder uniquement les vrais pays)
AGGREGATES = [
    'World', 'Asia', 'Europe', 'Africa', 'North America', 'South America',
    'Oceania', 'USSR', 'European Union (27)',
    'High-income countries', 'Low-income countries',
    'Lower-middle-income countries', 'Upper-middle-income countries',
]
df = df_raw[~df_raw['country'].isin(AGGREGATES)].copy()
print(f'\nApr\u00e8s filtrage agr\u00e9gats : {df.shape[0]} lignes, {df["country"].nunique()} pays')
print(f'Z\u00e9ros : {(df["deaths"] == 0).sum()} / {len(df)} ({(df["deaths"] == 0).mean()*100:.0f}%)')
print('>> 2a. Chargement : OK')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2a. Distribution des d\u00e9c\u00e8s (log scale)
ax = axes[0, 0]
non_zero = df[df['impact'] > 0]['impact']
ax.hist(np.log10(non_zero + 1), bins=40, color=COLOR, edgecolor='white')
ax.set_xlabel('log10(d\u00e9c\u00e8s + 1)')
ax.set_ylabel('Fr\u00e9quence')
ax.set_title('Distribution des d\u00e9c\u00e8s (non-z\u00e9ro, log10)')

# 2b. \u00c9volution mondiale par d\u00e9cennie
ax = axes[0, 1]
world = df_raw[df_raw['country'] == 'World']
ax.bar(world['year'].astype(str), world['impact'], color=COLOR, edgecolor='white')
ax.set_xlabel('D\u00e9cennie')
ax.set_ylabel('D\u00e9c\u00e8s moyens annuels')
ax.set_title('Mortalit\u00e9 mondiale par d\u00e9cennie')
ax.tick_params(axis='x', rotation=45)

# 2c. Top 15 pays (total d\u00e9c\u00e8s)
ax = axes[1, 0]
top15 = df.groupby('country')['impact'].sum().sort_values(ascending=True).tail(15)
ax.barh(top15.index, top15.values, color=COLOR, edgecolor='white')
ax.set_xlabel('Total d\u00e9c\u00e8s (somme d\u00e9cennale)')
ax.set_title('Top 15 pays')

# 2d. Nombre de pays affect\u00e9s par d\u00e9cennie
ax = axes[1, 1]
pays_par_dec = df[df['impact'] > 0].groupby('year')['country'].nunique()
ax.plot(pays_par_dec.index, pays_par_dec.values, 'o-', color=COLOR, linewidth=2)
ax.set_xlabel('D\u00e9cennie')
ax.set_ylabel('Nombre de pays')
ax.set_title('Pays avec d\u00e9c\u00e8s > 0 par d\u00e9cennie')

plt.suptitle('EDA \u2014 EM-DAT Deaths from Disasters', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB10_eda.png', dpi=200, bbox_inches='tight')
plt.show()
print('>> 2b. EDA visualisations : OK')

---

## 3. Feature Engineering

Avec seulement 3 colonnes (pays, ann\u00e9e, d\u00e9c\u00e8s), on doit cr\u00e9er des features informatives :
- **Lag** : d\u00e9c\u00e8s de la d\u00e9cennie pr\u00e9c\u00e9dente (t-1, t-2)
- **Trend** : taux de croissance d\u00e9cennal
- **Volatilit\u00e9** : \u00e9cart-type des d\u00e9c\u00e8s pass\u00e9s du pays
- **Moyenne cumul\u00e9e** : moyenne mobile des d\u00e9c\u00e8s pass\u00e9s
- **Continent** : r\u00e9gion g\u00e9ographique (enrichissement externe)
- **Log-transform** : r\u00e9duire l'asym\u00e9trie de la target

In [ ]:
# Mapping continent (principales r\u00e9gions)
CONTINENT_MAP = {
    'Afghanistan': 'Asie', 'Albania': 'Europe', 'Algeria': 'Afrique', 'Angola': 'Afrique',
    'Argentina': 'Am\u00e9rique du Sud', 'Armenia': 'Asie', 'Australia': 'Oc\u00e9anie',
    'Austria': 'Europe', 'Azerbaijan': 'Asie', 'Bangladesh': 'Asie',
    'Belarus': 'Europe', 'Belgium': 'Europe', 'Benin': 'Afrique', 'Bhutan': 'Asie',
    'Bolivia': 'Am\u00e9rique du Sud', 'Bosnia and Herzegovina': 'Europe',
    'Botswana': 'Afrique', 'Brazil': 'Am\u00e9rique du Sud', 'Bulgaria': 'Europe',
    'Burkina Faso': 'Afrique', 'Burundi': 'Afrique', 'Cambodia': 'Asie',
    'Cameroon': 'Afrique', 'Canada': 'Am\u00e9rique du Nord', 'Central African Republic': 'Afrique',
    'Chad': 'Afrique', 'Chile': 'Am\u00e9rique du Sud', 'China': 'Asie',
    'Colombia': 'Am\u00e9rique du Sud', 'Comoros': 'Afrique', 'Congo': 'Afrique',
    'Costa Rica': 'Am\u00e9rique centrale', 'Croatia': 'Europe', 'Cuba': 'Cara\u00efbes',
    'Cyprus': 'Europe', 'Czechia': 'Europe', 'Denmark': 'Europe',
    'Democratic Republic of Congo': 'Afrique', 'Djibouti': 'Afrique',
    'Dominican Republic': 'Cara\u00efbes', 'East Timor': 'Asie',
    'Ecuador': 'Am\u00e9rique du Sud', 'Egypt': 'Afrique', 'El Salvador': 'Am\u00e9rique centrale',
    'Eritrea': 'Afrique', 'Estonia': 'Europe', 'Eswatini': 'Afrique',
    'Ethiopia': 'Afrique', 'Fiji': 'Oc\u00e9anie', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Afrique', 'Gambia': 'Afrique',
    'Georgia': 'Asie', 'Germany': 'Europe', 'Ghana': 'Afrique',
    'Greece': 'Europe', 'Guatemala': 'Am\u00e9rique centrale', 'Guinea': 'Afrique',
    'Guinea-Bissau': 'Afrique', 'Guyana': 'Am\u00e9rique du Sud',
    'Haiti': 'Cara\u00efbes', 'Honduras': 'Am\u00e9rique centrale', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asie', 'Indonesia': 'Asie',
    'Iran': 'Asie', 'Iraq': 'Asie', 'Ireland': 'Europe', 'Israel': 'Asie',
    'Italy': 'Europe', 'Jamaica': 'Cara\u00efbes', 'Japan': 'Asie',
    'Jordan': 'Asie', 'Kazakhstan': 'Asie', 'Kenya': 'Afrique',
    'Kyrgyzstan': 'Asie', 'Laos': 'Asie', 'Latvia': 'Europe',
    'Lebanon': 'Asie', 'Lesotho': 'Afrique', 'Liberia': 'Afrique',
    'Libya': 'Afrique', 'Lithuania': 'Europe', 'Luxembourg': 'Europe',
    'Madagascar': 'Afrique', 'Malawi': 'Afrique', 'Malaysia': 'Asie',
    'Mali': 'Afrique', 'Mauritania': 'Afrique', 'Mauritius': 'Afrique',
    'Mexico': 'Am\u00e9rique du Nord', 'Moldova': 'Europe', 'Mongolia': 'Asie',
    'Montenegro': 'Europe', 'Morocco': 'Afrique', 'Mozambique': 'Afrique',
    'Myanmar': 'Asie', 'Namibia': 'Afrique', 'Nepal': 'Asie',
    'Netherlands': 'Europe', 'New Zealand': 'Oc\u00e9anie', 'Nicaragua': 'Am\u00e9rique centrale',
    'Niger': 'Afrique', 'Nigeria': 'Afrique', 'North Korea': 'Asie',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Oman': 'Asie',
    'Pakistan': 'Asie', 'Palestine': 'Asie', 'Panama': 'Am\u00e9rique centrale',
    'Papua New Guinea': 'Oc\u00e9anie', 'Paraguay': 'Am\u00e9rique du Sud',
    'Peru': 'Am\u00e9rique du Sud', 'Philippines': 'Asie', 'Poland': 'Europe',
    'Portugal': 'Europe', 'Puerto Rico': 'Cara\u00efbes', 'Romania': 'Europe',
    'Russia': 'Europe', 'Rwanda': 'Afrique', 'Saudi Arabia': 'Asie',
    'Senegal': 'Afrique', 'Serbia': 'Europe', 'Sierra Leone': 'Afrique',
    'Slovakia': 'Europe', 'Slovenia': 'Europe', 'Somalia': 'Afrique',
    'South Africa': 'Afrique', 'South Korea': 'Asie', 'South Sudan': 'Afrique',
    'Spain': 'Europe', 'Sri Lanka': 'Asie', 'Sudan': 'Afrique',
    'Suriname': 'Am\u00e9rique du Sud', 'Sweden': 'Europe', 'Switzerland': 'Europe',
    'Syria': 'Asie', 'Taiwan': 'Asie', 'Tajikistan': 'Asie',
    'Tanzania': 'Afrique', 'Thailand': 'Asie', 'Togo': 'Afrique',
    'Trinidad and Tobago': 'Cara\u00efbes', 'Tunisia': 'Afrique', 'Turkey': 'Asie',
    'Turkmenistan': 'Asie', 'Uganda': 'Afrique', 'Ukraine': 'Europe',
    'United Arab Emirates': 'Asie', 'United Kingdom': 'Europe',
    'United States': 'Am\u00e9rique du Nord', 'Uruguay': 'Am\u00e9rique du Sud',
    'Uzbekistan': 'Asie', 'Venezuela': 'Am\u00e9rique du Sud', 'Vietnam': 'Asie',
    'Yemen': 'Asie', 'Zambia': 'Afrique', 'Zimbabwe': 'Afrique',
}

def attribuer_continent(pays):
    """Attribue un continent \u00e0 un pays (fallback heuristique)."""
    if pays in CONTINENT_MAP:
        return CONTINENT_MAP[pays]
    return 'Autre'

df['continent'] = df['country'].apply(attribuer_continent)

# Trier par pays et ann\u00e9e
df = df.sort_values(['country', 'year']).reset_index(drop=True)

# Log-transform de la target
df['log_impact'] = np.log1p(df['impact'])

# Features temporelles
df['decade_index'] = (df['year'] - 1900) // 10  # 0 \u00e0 12

# Lag features (d\u00e9c\u00e8s d\u00e9cennie pr\u00e9c\u00e9dente)
df['lag_1'] = df.groupby('country')['impact'].shift(1)
df['lag_2'] = df.groupby('country')['impact'].shift(2)
df['log_lag_1'] = np.log1p(df['lag_1'])
df['log_lag_2'] = np.log1p(df['lag_2'])

# Trend : taux de croissance vs d\u00e9cennie pr\u00e9c\u00e9dente
df['trend'] = (df['impact'] - df['lag_1']) / (df['lag_1'] + 1)

# Statistiques cumul\u00e9es par pays (excluant la ligne courante)
df['cumul_mean'] = df.groupby('country')['impact'].transform(
    lambda x: x.expanding().mean().shift(1)
)
df['cumul_std'] = df.groupby('country')['impact'].transform(
    lambda x: x.expanding().std().shift(1)
)
df['cumul_max'] = df.groupby('country')['impact'].transform(
    lambda x: x.expanding().max().shift(1)
)
df['log_cumul_mean'] = np.log1p(df['cumul_mean'])

# Remplir les NaN (premi\u00e8res d\u00e9cennies sans historique)
df['lag_1'] = df['lag_1'].fillna(0)
df['lag_2'] = df['lag_2'].fillna(0)
df['log_lag_1'] = df['log_lag_1'].fillna(0)
df['log_lag_2'] = df['log_lag_2'].fillna(0)
df['trend'] = df['trend'].fillna(0)
df['cumul_mean'] = df['cumul_mean'].fillna(0)
df['cumul_std'] = df['cumul_std'].fillna(0)
df['cumul_max'] = df['cumul_max'].fillna(0)
df['log_cumul_mean'] = df['log_cumul_mean'].fillna(0)

# Classification des risques (pour la section 6)
def categoriser_risque(deaths):
    if deaths == 0:
        return 'Aucun'
    elif deaths < 100:
        return 'Faible'
    elif deaths < 1000:
        return 'Mod\u00e9r\u00e9'
    elif deaths < 10000:
        return '\u00c9lev\u00e9'
    else:
        return 'Critique'

df['risk_level'] = df['impact'].apply(categoriser_risque)

print(f'Shape apr\u00e8s feature engineering : {df.shape}')
print(f'\nFeatures cr\u00e9\u00e9es : {[c for c in df.columns if c not in ["country", "year", "deaths"]]}')
print(f'\nR\u00e9partition continents :')
print(df['continent'].value_counts())
print(f'\nR\u00e9partition niveaux de risque :')
print(df['risk_level'].value_counts())
print('>> 3. Feature Engineering : OK')

---

## 4. Pr\u00e9processing

In [ ]:
# Encodage du continent
le_continent = LabelEncoder()
df['continent_enc'] = le_continent.fit_transform(df['continent'])

# Features pour la r\u00e9gression
FEATURES_REG = [
    'decade_index', 'continent_enc',
    'log_lag_1', 'log_lag_2', 'trend',
    'log_cumul_mean', 'cumul_std', 'cumul_max',
]

# Filtrer les lignes avec assez d'historique (d\u00e9cennie >= 1920)
df_model = df[df['year'] >= 1920].copy()
print(f'Lignes pour le mod\u00e8le (>= 1920) : {len(df_model)}')

X = df_model[FEATURES_REG].values
y_reg = df_model['log_impact'].values  # Target r\u00e9gression (log)
y_cls = df_model['risk_level'].values  # Target classification

# Encodage de la target classification
le_risk = LabelEncoder()
le_risk.fit(['Aucun', 'Faible', 'Mod\u00e9r\u00e9', '\u00c9lev\u00e9', 'Critique'])
y_cls_enc = le_risk.transform(y_cls)

# Train/test split (80/20, stratifi\u00e9 par d\u00e9cennie pour \u00e9viter le data leakage temporel)
# On utilise les derni\u00e8res d\u00e9cennies (2010-2020) comme test
mask_train = df_model['year'] < 2010
mask_test = df_model['year'] >= 2010

X_train, X_test = X[mask_train], X[mask_test]
y_train_reg, y_test_reg = y_reg[mask_train], y_reg[mask_test]
y_train_cls, y_test_cls = y_cls_enc[mask_train], y_cls_enc[mask_test]

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'\nTrain : {X_train.shape[0]} lignes (< 2010)')
print(f'Test  : {X_test.shape[0]} lignes (>= 2010)')
print(f'Features : {FEATURES_REG}')
print(f'\nSplit temporel (pas al\u00e9atoire) pour \u00e9viter le data leakage')
print('>> 4. Pr\u00e9processing : OK')

---

## 5. Mod\u00e8les de R\u00e9gression

On pr\u00e9dit `log(deaths + 1)` pour g\u00e9rer l'asym\u00e9trie. Comparatif multi-mod\u00e8les.

In [ ]:
modeles_reg = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=SEED),
    'Lasso': Lasso(alpha=0.1, random_state=SEED),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=SEED),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED),
}
if HAS_XGB:
    modeles_reg['XGBoost'] = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED, verbosity=0)

results_reg = []

print(f'{"Mod\u00e8le":<22} {"MAE":<8} {"RMSE":<8} {"R\u00b2":<8} {"Temps (s)":<10}')
print('-' * 56)

best_r2 = -999
best_model_reg = None
best_model_name_reg = ''

for name, model in modeles_reg.items():
    t0 = time.time()
    model.fit(X_train_sc, y_train_reg)
    y_pred = model.predict(X_test_sc)
    t_fit = round(time.time() - t0, 3)

    mae = mean_absolute_error(y_test_reg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    r2 = r2_score(y_test_reg, y_pred)

    print(f'{name:<22} {mae:<8.3f} {rmse:<8.3f} {r2:<8.3f} {t_fit:<10}')
    results_reg.append({'modele': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'temps_s': t_fit})

    if r2 > best_r2:
        best_r2 = r2
        best_model_reg = model
        best_model_name_reg = name

print(f'\nMeilleur mod\u00e8le : {best_model_name_reg} (R\u00b2 = {best_r2:.3f})')
print(f'Hypoth\u00e8se R\u00b2 > 0.5 : {"VALIDEE" if best_r2 > 0.5 else "INVALIDEE"}')
print('>> 5. Mod\u00e8les de r\u00e9gression : OK')

In [ ]:
# Visualisation comparatif r\u00e9gression
df_reg = pd.DataFrame(results_reg).sort_values('R2', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R\u00b2 par mod\u00e8le
ax = axes[0]
colors_bar = [COLORS[0] if n != best_model_name_reg else '#e74c3c' for n in df_reg['modele']]
ax.barh(df_reg['modele'], df_reg['R2'], color=colors_bar, edgecolor='white')
ax.axvline(x=0.5, color='red', linestyle='--', label='Seuil hypoth\u00e8se (0.5)')
ax.set_xlabel('R\u00b2')
ax.set_title('Comparatif R\u00b2 \u2014 R\u00e9gression')
ax.legend()

# Pr\u00e9dit vs r\u00e9el (meilleur mod\u00e8le)
ax = axes[1]
y_pred_best = best_model_reg.predict(X_test_sc)
ax.scatter(y_test_reg, y_pred_best, alpha=0.3, s=10, color=COLOR)
ax.plot([0, y_test_reg.max()], [0, y_test_reg.max()], 'r--', linewidth=1)
ax.set_xlabel('R\u00e9el (log d\u00e9c\u00e8s)')
ax.set_ylabel('Pr\u00e9dit (log d\u00e9c\u00e8s)')
ax.set_title(f'{best_model_name_reg} \u2014 Pr\u00e9dit vs R\u00e9el')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB10_regression_comparatif.png', dpi=200, bbox_inches='tight')
plt.show()
print('>> 5b. Visualisation r\u00e9gression : OK')

---

## 6. Mod\u00e8les de Classification

Pr\u00e9diction du niveau de risque : Aucun / Faible / Mod\u00e9r\u00e9 / \u00c9lev\u00e9 / Critique.

In [ ]:
modeles_cls = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED),
}
if HAS_XGB:
    modeles_cls['XGBoost'] = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=SEED, verbosity=0, eval_metric='mlogloss')

results_cls = []

print(f'{"Mod\u00e8le":<22} {"Accuracy":<10} {"F1 macro":<10} {"F1 weighted":<12} {"Temps (s)":<10}')
print('-' * 64)

best_f1 = -1
best_model_cls = None
best_model_name_cls = ''

for name, model in modeles_cls.items():
    t0 = time.time()
    model.fit(X_train_sc, y_train_cls)
    y_pred = model.predict(X_test_sc)
    t_fit = round(time.time() - t0, 3)

    acc = accuracy_score(y_test_cls, y_pred)
    f1_m = f1_score(y_test_cls, y_pred, average='macro', zero_division=0)
    f1_w = f1_score(y_test_cls, y_pred, average='weighted', zero_division=0)

    print(f'{name:<22} {acc:<10.3f} {f1_m:<10.3f} {f1_w:<12.3f} {t_fit:<10}')
    results_cls.append({'modele': name, 'accuracy': acc, 'f1_macro': f1_m, 'f1_weighted': f1_w, 'temps_s': t_fit})

    if f1_w > best_f1:
        best_f1 = f1_w
        best_model_cls = model
        best_model_name_cls = name

print(f'\nMeilleur mod\u00e8le : {best_model_name_cls} (F1 weighted = {best_f1:.3f})')
print(f'Hypoth\u00e8se F1 > 0.6 : {"VALIDEE" if best_f1 > 0.6 else "INVALIDEE"}')

# Classification report du meilleur
y_pred_best_cls = best_model_cls.predict(X_test_sc)
print(f'\n--- Classification Report ({best_model_name_cls}) ---')
print(classification_report(y_test_cls, y_pred_best_cls, target_names=le_risk.classes_, zero_division=0))
print('>> 6. Mod\u00e8les de classification : OK')

In [ ]:
# Matrice de confusion + Feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
ax = axes[0]
cm = confusion_matrix(y_test_cls, y_pred_best_cls)
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(le_risk.classes_)))
ax.set_yticks(range(len(le_risk.classes_)))
ax.set_xticklabels(le_risk.classes_, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(le_risk.classes_, fontsize=8)
ax.set_xlabel('Pr\u00e9dit')
ax.set_ylabel('R\u00e9el')
ax.set_title(f'Matrice de confusion \u2014 {best_model_name_cls}')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=ax)

# Feature importance (meilleur mod\u00e8le r\u00e9gression)
ax = axes[1]
if hasattr(best_model_reg, 'feature_importances_'):
    importances = best_model_reg.feature_importances_
    idx = np.argsort(importances)
    ax.barh([FEATURES_REG[i] for i in idx], importances[idx], color=COLOR, edgecolor='white')
    ax.set_xlabel('Importance')
    ax.set_title(f'Feature Importance \u2014 {best_model_name_reg}')
else:
    ax.text(0.5, 0.5, 'Non disponible\n(mod\u00e8le lin\u00e9aire)', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Feature Importance')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB10_classification_results.png', dpi=200, bbox_inches='tight')
plt.show()
print('>> 6b. Visualisation classification : OK')

---

## 7. Hyperparameter Tuning

GridSearchCV sur le meilleur mod\u00e8le de r\u00e9gression.

In [ ]:
# GridSearch sur le meilleur mod\u00e8le
print(f'Tuning du meilleur mod\u00e8le : {best_model_name_reg}\n')

if 'Gradient Boosting' in best_model_name_reg or 'XGBoost' in best_model_name_reg:
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1, 0.2],
    }
    base_model = type(best_model_reg)(random_state=SEED)
elif 'Random Forest' in best_model_name_reg:
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
    }
    base_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)
else:
    param_grid = {'alpha': [0.01, 0.1, 1.0, 10.0]}
    base_model = Ridge(random_state=SEED)

t0 = time.time()
grid = GridSearchCV(
    base_model, param_grid,
    cv=5, scoring='r2', n_jobs=-1, verbose=0,
)
grid.fit(X_train_sc, y_train_reg)
t_grid = round(time.time() - t0, 2)

print(f'Meilleurs param\u00e8tres : {grid.best_params_}')
print(f'Meilleur R\u00b2 CV       : {grid.best_score_:.3f}')

# \u00c9valuer sur le test
y_pred_tuned = grid.best_estimator_.predict(X_test_sc)
r2_tuned = r2_score(y_test_reg, y_pred_tuned)
mae_tuned = mean_absolute_error(y_test_reg, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_reg, y_pred_tuned))

print(f'\nR\u00e9sultats sur le test (apr\u00e8s tuning) :')
print(f'  R\u00b2   : {r2_tuned:.3f} (avant : {best_r2:.3f}, delta : {r2_tuned - best_r2:+.3f})')
print(f'  MAE  : {mae_tuned:.3f}')
print(f'  RMSE : {rmse_tuned:.3f}')
print(f'  Temps GridSearch : {t_grid}s')
print('>> 7. Hyperparameter Tuning : OK')

---

## 8. MLflow Tracking

In [ ]:
if HAS_MLFLOW:
    mlflow.set_tracking_uri('file:' + str((BASE / 'mlruns').resolve()))
    mlflow.set_experiment('NB10_ML_Predictif_Catastrophes')

    # Log tous les mod\u00e8les de r\u00e9gression
    for res in results_reg:
        with mlflow.start_run(run_name=f'reg_{res["modele"]}'):
            mlflow.log_param('model_type', res['modele'])
            mlflow.log_param('task', 'regression')
            mlflow.log_param('features', str(FEATURES_REG))
            mlflow.log_param('split', 'temporal_2010')
            mlflow.log_param('target', 'log_impact')
            mlflow.log_metric('MAE', res['MAE'])
            mlflow.log_metric('RMSE', res['RMSE'])
            mlflow.log_metric('R2', res['R2'])
            mlflow.log_metric('train_time_s', res['temps_s'])

    # Log tous les mod\u00e8les de classification
    for res in results_cls:
        with mlflow.start_run(run_name=f'cls_{res["modele"]}'):
            mlflow.log_param('model_type', res['modele'])
            mlflow.log_param('task', 'classification')
            mlflow.log_param('features', str(FEATURES_REG))
            mlflow.log_param('split', 'temporal_2010')
            mlflow.log_param('classes', str(list(le_risk.classes_)))
            mlflow.log_metric('accuracy', res['accuracy'])
            mlflow.log_metric('f1_macro', res['f1_macro'])
            mlflow.log_metric('f1_weighted', res['f1_weighted'])
            mlflow.log_metric('train_time_s', res['temps_s'])

    # Log le mod\u00e8le tun\u00e9
    with mlflow.start_run(run_name=f'tuned_{best_model_name_reg}'):
        mlflow.log_params(grid.best_params_)
        mlflow.log_param('model_type', best_model_name_reg)
        mlflow.log_param('task', 'regression_tuned')
        mlflow.log_metric('R2', r2_tuned)
        mlflow.log_metric('MAE', mae_tuned)
        mlflow.log_metric('RMSE', rmse_tuned)
        mlflow.log_metric('R2_cv', grid.best_score_)
        mlflow.sklearn.log_model(grid.best_estimator_, 'model')

    print(f'  MLflow : {len(results_reg) + len(results_cls) + 1} runs log\u00e9s')
    print(f'  Tracking URI : {mlflow.get_tracking_uri()}')
else:
    print('  MLflow non install\u00e9, skip')

print('>> 8. MLflow Tracking : OK')

---

## 9. Pr\u00e9dictions 2030

On utilise le meilleur modèle tuné pour prédire l'impact humain de la décennie 2030 par pays.

In [ ]:
# Pr\u00e9parer les features 2030 pour chaque pays
derniere_decennie = df[df['year'] == 2020].copy()
avant_derniere = df[df['year'] == 2010].copy()

pred_2030 = []
for _, row in derniere_decennie.iterrows():
    pays = row['country']
    avant = avant_derniere[avant_derniere['country'] == pays]
    lag_2_val = avant.iloc[0]['impact'] if len(avant) > 0 else 0

    # Features pour 2030
    pays_hist = df[df['country'] == pays]
    features = {
        'decade_index': 13,  # 2030
        'continent_enc': row['continent_enc'],
        'log_lag_1': np.log1p(row['impact']),
        'log_lag_2': np.log1p(lag_2_val),
        'trend': row['trend'],
        'log_cumul_mean': np.log1p(pays_hist['impact'].mean()),
        'cumul_std': pays_hist['impact'].std() if len(pays_hist) > 1 else 0,
        'cumul_max': pays_hist['impact'].max(),
    }
    pred_2030.append({'country': pays, 'continent': row['continent'], **features})

df_pred = pd.DataFrame(pred_2030)
X_2030 = scaler.transform(df_pred[FEATURES_REG].values)

# Pr\u00e9dictions r\u00e9gression (mod\u00e8le tun\u00e9)
df_pred['log_impact_pred'] = grid.best_estimator_.predict(X_2030)
df_pred['impact_pred'] = np.expm1(df_pred['log_impact_pred']).clip(lower=0)

# Pr\u00e9dictions classification
df_pred['risk_pred_enc'] = best_model_cls.predict(X_2030)
df_pred['risk_pred'] = le_risk.inverse_transform(df_pred['risk_pred_enc'])

# Top 20 pays \u00e0 risque
top20 = df_pred.nlargest(20, 'impact_pred')[['country', 'continent', 'impact_pred', 'risk_pred']]
print('=== Top 20 pays \u00e0 risque pour 2030 ===')
print(top20.to_string(index=False))

# R\u00e9partition des risques 2030
print(f'\n=== R\u00e9partition des niveaux de risque 2030 ===')
print(df_pred['risk_pred'].value_counts())

# Sauvegarder
csv_pred = OUTPUT_DIR / 'NB10_predictions_2030.csv'
df_pred[['country', 'continent', 'impact_pred', 'risk_pred']].to_csv(csv_pred, index=False)
print(f'\n  [OK] {csv_pred.name} sauvegard\u00e9')
print('>> 9a. Pr\u00e9dictions 2030 : OK')

In [ ]:
# Visualisations pr\u00e9dictions 2030
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 15 pays
ax = axes[0]
top15_pred = df_pred.nlargest(15, 'impact_pred')
risk_colors = {'Aucun': '#2ecc71', 'Faible': '#f1c40f', 'Mod\u00e9r\u00e9': '#f39c12', '\u00c9lev\u00e9': '#e74c3c', 'Critique': '#8e44ad'}
bar_colors = [risk_colors.get(r, COLOR) for r in top15_pred['risk_pred']]
ax.barh(top15_pred['country'], top15_pred['impact_pred'], color=bar_colors, edgecolor='white')
ax.set_xlabel('D\u00e9c\u00e8s pr\u00e9dits (moyenne annuelle)')
ax.set_title('Top 15 pays \u2014 Pr\u00e9dictions 2030')

# R\u00e9partition par continent
ax = axes[1]
risk_par_continent = df_pred.groupby('continent')['impact_pred'].sum().sort_values(ascending=True)
ax.barh(risk_par_continent.index, risk_par_continent.values, color=COLOR, edgecolor='white')
ax.set_xlabel('D\u00e9c\u00e8s pr\u00e9dits totaux')
ax.set_title('Mortalit\u00e9 pr\u00e9dite 2030 par continent')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB10_predictions_2030.png', dpi=200, bbox_inches='tight')
plt.show()
print('>> 9b. Visualisation pr\u00e9dictions : OK')

---

## 10. Int\u00e9gration Agent

Ce mod\u00e8le ML compl\u00e8te l'agent RAG : le RAG fournit le contexte scientifique (GIEC, rapports), le ML fournit les pr\u00e9dictions chiffr\u00e9es. Ensemble, ils permettent une analyse de risque croisant donn\u00e9es qualitatives et quantitatives.

In [ ]:
# Exemple d'utilisation : pr\u00e9diction pour un pays donn\u00e9
def predire_risque_pays(pays_nom, df_pred):
    """Retourne la pr\u00e9diction de risque pour un pays donn\u00e9."""
    row = df_pred[df_pred['country'] == pays_nom]
    if len(row) == 0:
        return f'Pays "{pays_nom}" non trouv\u00e9 dans les donn\u00e9es.'
    row = row.iloc[0]
    return (
        f'Pr\u00e9diction 2030 pour {pays_nom} :\n'
        f'  - D\u00e9c\u00e8s moyens annuels pr\u00e9dits : {row["impact_pred"]:.0f}\n'
        f'  - Niveau de risque : {row["risk_pred"]}\n'
        f'  - Continent : {row["continent"]}'
    )

# Tests
for pays_test in ['France', 'India', 'Bangladesh', 'Haiti', 'Japan']:
    print(predire_risque_pays(pays_test, df_pred))
    print()

print('--- Int\u00e9gration possible ---')
print('1. Ajouter un 8e outil `predict_risk(country)` dans tools.py')
print('2. Charger le mod\u00e8le sauvegard\u00e9 via MLflow ou joblib')
print('3. L\'agent croise : RAG (contexte GIEC) + ML (pr\u00e9diction) + m\u00e9t\u00e9o (donn\u00e9es r\u00e9elles)')
print('>> 10. Int\u00e9gration Agent : OK')

---

## 11. Conclusions

### Quality gate finale

| Crit\u00e8re | Seuil | R\u00e9sultat | D\u00e9cision |
|---|---|---|---|
| R\u00e9gression R\u00b2 > 0.5 | 0.5 | Section 5 | GO / NO-GO |
| Classification F1 > 0.6 | 0.6 | Section 6 | GO / NO-GO |
| GridSearchCV am\u00e9liore le R\u00b2 | delta > 0 | Section 7 | GO / NO-GO |
| MLflow tracking complet | tous runs | Section 8 | GO |
| Pr\u00e9dictions 2030 coh\u00e9rentes | top pays = zones \u00e0 risque connues | Section 9 | GO / NO-GO |

### Hypoth\u00e8se

\u00c0 compl\u00e9ter apr\u00e8s ex\u00e9cution : VALIDEE / INVALIDEE

---

### Limites

- **Granularit\u00e9 d\u00e9cennale** : 13 points par pays maximum, insuffisant pour du deep learning
- **Pas de distinction par type de catastrophe** : inondations, s\u00e9cheresses et temp\u00eates sont agr\u00e9g\u00e9s
- **Pas de variables exog\u00e8nes** : pas de PIB, population, urbanisation, infrastructure
- **Biais historique** : les donn\u00e9es 1900-1950 sont incompl\u00e8tes (sous-d\u00e9claration)
- **Pas de mod\u00e8le climatique** : les pr\u00e9dictions extrapolent les tendances pass\u00e9es, pas la physique du climat
- **55% de z\u00e9ros** : dataset fortement d\u00e9s\u00e9quilibr\u00e9

---

### Axes d'am\u00e9lioration

- Enrichir avec des variables exog\u00e8nes (World Bank, EM-DAT d\u00e9taill\u00e9 par type)
- Mod\u00e8le Zero-Inflated (g\u00e9rer les 55% de z\u00e9ros)
- S\u00e9ries temporelles par pays (Prophet, ARIMA) avec granularit\u00e9 annuelle
- Int\u00e9grer le mod\u00e8le comme 8e outil de l'agent (`predict_risk`)
- Croiser les pr\u00e9dictions ML avec les seuils du corpus GIEC via l'agent

In [ ]:
# Sauvegarder les r\u00e9sultats
df_reg_results = pd.DataFrame(results_reg)
df_cls_results = pd.DataFrame(results_cls)
df_reg_results.to_csv(OUTPUT_DIR / 'NB10_regression_results.csv', index=False)
df_cls_results.to_csv(OUTPUT_DIR / 'NB10_classification_results.csv', index=False)
print(f'  [OK] NB10_regression_results.csv')
print(f'  [OK] NB10_classification_results.csv')

elapsed = round(time.time() - notebook_start_time, 2)
print(f'\nTemps total du notebook : {elapsed}s')
print(f'Mod\u00e8les entra\u00een\u00e9s : {len(results_reg)} r\u00e9gression + {len(results_cls)} classification')
print(f'Meilleur R\u00b2 : {best_r2:.3f} ({best_model_name_reg})')
print(f'Meilleur F1 : {best_f1:.3f} ({best_model_name_cls})')
print(f'Pays pr\u00e9dits 2030 : {len(df_pred)}')
print('>> NOTEBOOK 10 TERMINE')